# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source

The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata
metadata = dataset.metadata.to_json()
print(f"Name: {metadata['name']}")
print(f"Description: {metadata['description']}")
print(f"Identifier: {metadata['identifier']}")
print(f"Published: {metadata['datePublished']}")
print(f"License: {metadata['license']}")
print(f"Keywords: {metadata['keywords']}")

## 2. Data Overview

Review available record sets, fields, and their IDs.

Entities in the Croissant schema are referenced by their `@id` values. Let's enumerate record sets and their fields.

In [ ]:
# List record sets and their fields using @id
record_sets = dataset.record_sets
print("Available Record Sets:")
for rs in record_sets:
    print(f"- Name: {rs.name}, @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    * {field.name} (@id: {field.id}, data_type: {field.data_type})")
    print()

# For preview, print the first record set's records
if record_sets:
    first_rs_id = record_sets[0].id
    print(f"Sample records from record set @id: {first_rs_id}")
    for record in dataset.records(record_set=first_rs_id):
        print(record)
        break  # print only the first record as a sample

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

For this example, we will extract all record sets available and load them to pandas DataFrames using their `@id`.

In [ ]:
# Extract data per record set
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Print available DataFrame columns for first record set
if record_set_ids:
    print(f"Columns for record set @id {record_set_ids[0]}:")
    print(dataframes[record_set_ids[0]].columns.tolist())
    display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 
Operations like removing outliers, transforming data distributions, or grouping data by key attributes prepare it for further analysis.

Refer to all fields and columns using their `@id`.

In [ ]:
# Example EDA for the main clinical record set
main_record_set_id = record_set_ids[0]
df = dataframes[main_record_set_id]

# List columns and select numeric and grouping fields by @id
print("Columns in main record set:")
print(df.columns.tolist())

# For demonstration, suppose there is a numeric column with @id 'age' and grouping field @id 'sex'
# Replace with real @ids as found in previous overview
numeric_field_id = None
group_field_id = None
for col in df.columns:
    if 'age' in col.lower():
        numeric_field_id = col
    if 'sex' in col.lower() or 'gender' in col.lower():
        group_field_id = col

if numeric_field_id is not None:
    threshold = 50
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())
    
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group
    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric field found for EDA (try updating with the correct @id).")

## 5. Visualization

Visualize data distributions or relationships between fields. For example, plot the age distribution or anatomical location frequencies.

Use `matplotlib` and reference columns by their `@id`.

In [ ]:
import matplotlib.pyplot as plt

# Plot numeric field distribution
if numeric_field_id is not None:
    plt.figure(figsize=(8,6))
    df[numeric_field_id].hist(bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# Plot group field frequencies
if group_field_id is not None:
    plt.figure(figsize=(6,4))
    df[group_field_id].value_counts().plot(kind='bar')
    plt.title(f"Counts by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel("Number of Records")
    plt.show()

## 6. Conclusion

In this notebook, we loaded the FAIR^2 Clinicopathological dataset via its Croissant schema, explored its structure and fields using their `@id`s, and performed initial exploratory analysis and visualizations. 

- Referencing by `@id` ensures reproducibility and clarity.
- The dataset contains rich clinical and molecular data suitable for stratified analyses in cancer survivors.
- The mlcroissant library enables seamless loading and manipulation of FAIR datasets.

Further analysis may include advanced statistical modeling, feature engineering, and integration with clinical outcome variables.